# Crime_Complaints_And_Arrests_NYC_Data_Project

### 1. What percentage of 2025 arrests involved each sex?
### 2. What percentage of 2025 arrests involved each race?
### 3. How do race arrest rates differ by borough?
### 4. How do arrests compare with complaints across NYC neighborhoods?


In [ ]:
import pandas as pd 
import seaborn as sns
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
complaints = pd.read_csv("complaints_2025.csv")
complaints

In [ ]:
arrests = pd.read_csv("arrests_2025.csv")
arrests

## Data cleaning

In [ ]:
arrests[["arrest_date", "perp_sex", "perp_race"]].dtypes

In [ ]:
arrests["arrest_date"] = pd.to_datetime(arrests["arrest_date"])
arrests["perp_sex"] = arrests["perp_sex"].replace({"M": "Male", "F": "Female", "(null)": "Unknown"})
arrests["perp_race"] = arrests["perp_race"].replace({"UNKNOWN": "Unknown", "(null)": "Unknown"})

## Arrests by race

In [ ]:
race = arrests.loc[arrests["perp_race"] != "Unknown", "perp_race"].str.title()
race_counts = race.value_counts().rename_axis("Race").reset_index(name="Count")

In [ ]:
race_counts["Percentage"] = race_counts["Count"] / race_counts["Count"].sum() * 100

race_counts

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

ax = sns.barplot(data=race_counts, x="Percentage", y="Race", color="#4DABF7")
labels = [f"{percentage:.1f}% ({count:,})" for percentage, count in zip(race_counts["Percentage"], race_counts["Count"])]
ax.bar_label(ax.containers[0], labels=labels, padding=3)

ax.set(
    title="Percentage of 2025 Arrests by Race",
    xlabel="Percentage of Arrests",
    ylabel="Race",
    xlim=(0, race_counts["Percentage"].max() + 5)
)
plt.tight_layout()
plt.show()

## Arrests by sex

*Note: 147,237 arrests (52.78%) had unknown sex and were excluded.*

In [ ]:
sex = arrests.loc[arrests["perp_sex"] != "Unknown", "perp_sex"]
sex_counts = sex.value_counts().rename_axis("Sex").reset_index(name="Count")

In [ ]:
sex_counts["Percentage"] = sex_counts["Count"] / sex_counts["Count"].sum() * 100

sex_counts

In [ ]:
plt.figure(figsize=(8, 6))

ax = sns.barplot(data=sex_counts, x="Sex", y="Percentage", color="#63E6BE")
labels = [f"{percentage:.1f}% ({count:,})" for percentage, count in zip(sex_counts["Percentage"], sex_counts["Count"])]
ax.bar_label(ax.containers[0], labels=labels, padding=3)

ax.set(
    title="Percentage of 2025 Arrests by Sex",
    xlabel="Sex",
    ylabel="Percentage of Arrests",
    ylim=(0, sex_counts["Percentage"].max() + 10)
)
plt.tight_layout()
plt.show()

## 2025 population by race and borough

Source: [U.S. Census Bureau Vintage 2025 Population Estimates](https://www.census.gov/programs-surveys/popest/data/tables.html)

In [ ]:
census_url = "https://www2.census.gov/programs-surveys/popest/datasets/2020-2025/counties/asrh/cc-est2025-alldata-36.csv"
census = pd.read_csv(census_url, encoding="latin-1")

In [ ]:
borough_counties = [5, 47, 61, 81, 85]

borough_population = census.query("COUNTY in @borough_counties and YEAR == 7 and AGEGRP == 0").copy()

In [ ]:
borough_names = {5: "Bronx", 47: "Brooklyn", 61: "Manhattan", 81: "Queens", 85: "Staten Island"}
borough_population["Borough"] = borough_population["COUNTY"].map(borough_names)

In [ ]:
race_population = pd.DataFrame()
race_population["Borough"] = borough_population["Borough"]

In [ ]:
race_population["BLACK"] = borough_population["NHBA_MALE"] + borough_population["NHBA_FEMALE"]
race_population["WHITE"] = borough_population["NHWA_MALE"] + borough_population["NHWA_FEMALE"]

In [ ]:
race_population["WHITE HISPANIC"] = borough_population["HWA_MALE"] + borough_population["HWA_FEMALE"]
race_population["BLACK HISPANIC"] = borough_population["HBA_MALE"] + borough_population["HBA_FEMALE"]

In [ ]:
race_population["ASIAN / PACIFIC ISLANDER"] = borough_population["AA_MALE"] + borough_population["AA_FEMALE"] + borough_population["NA_MALE"] + borough_population["NA_FEMALE"]
race_population["AMERICAN INDIAN/ALASKAN NATIVE"] = borough_population["IA_MALE"] + borough_population["IA_FEMALE"]

In [ ]:
race_population = race_population.melt(id_vars="Borough", var_name="Race", value_name="Population")

## Race arrest rates by borough

In [ ]:
borough_codes = {"B": "Bronx", "K": "Brooklyn", "M": "Manhattan", "Q": "Queens", "S": "Staten Island"}
arrests["Borough"] = arrests["arrest_boro"].map(borough_codes)

race_arrests = arrests[arrests["perp_race"] != "Unknown"]
race_arrests = race_arrests.groupby(["Borough", "perp_race"]).size().reset_index(name="Arrests")
race_arrests.columns = ["Borough", "Race", "Arrests"]

In [ ]:
race_rates = race_arrests.merge(race_population, on=["Borough", "Race"])
race_rates["Arrests per 100,000"] = race_rates["Arrests"] / race_rates["Population"] * 100000

race_rates

In [ ]:
race_heatmap = race_rates.pivot(index="Race", columns="Borough", values="Arrests per 100,000")
race_heatmap = race_heatmap[["Bronx", "Brooklyn", "Manhattan", "Queens", "Staten Island"]]

plt.figure(figsize=(12, 7))
sns.heatmap(race_heatmap, annot=True, fmt=".0f", cmap="YlOrRd")

plt.title("Race Arrest Rates by Borough in 2025")
plt.xlabel("Borough")
plt.ylabel("Race")
plt.tight_layout()
plt.show()

## 2025 arrests by NYC neighborhood

*The map shows where arrests happened, not where the arrested people live.*

In [ ]:
nta = gpd.read_file("nynta2020_26b/nynta2020.shp")

In [ ]:
arrest_points = arrests.dropna(subset=["latitude", "longitude"]).copy()

In [ ]:
arrest_points = gpd.GeoDataFrame(
    arrest_points,
    geometry=gpd.points_from_xy(
        arrest_points["longitude"],
        arrest_points["latitude"]
    ),
    crs="EPSG:4326"
)

arrest_points = arrest_points.to_crs(nta.crs)

In [ ]:
arrest_nta = gpd.sjoin(
    arrest_points,
    nta[["NTA2020", "NTAName", "geometry"]],
    how="inner",
    predicate="within"
)

In [ ]:
arrest_counts = arrest_nta.groupby("NTA2020")["arrest_key"].nunique()
arrest_counts = arrest_counts.reset_index(name="arrest_count")

In [ ]:
arrest_map = nta.merge(arrest_counts, on="NTA2020", how="left")
arrest_map["arrest_count"] = arrest_map["arrest_count"].fillna(0)

In [ ]:
ax = arrest_map.plot(
    column="arrest_count",
    figsize=(15, 15),
    cmap="Blues",
    legend=True,
    edgecolor="black",
    linewidth=0.2,
    legend_kwds={"label": "Number of Arrests"}
)

ax.set_title("2025 Arrests by NYC NTA", fontsize=18)
ax.axis("off")
plt.show()

## Arrest share compared with complaint share

*Red areas have a larger share of arrests. Blue areas have a larger share of complaints. This does not show which complaints led to arrests.*

In [ ]:
complaint_points = complaints.dropna(subset=["latitude", "longitude"]).copy()

In [ ]:
complaint_points = gpd.GeoDataFrame(
    complaint_points,
    geometry=gpd.points_from_xy(
        complaint_points["longitude"],
        complaint_points["latitude"]
    ),
    crs="EPSG:4326"
)

complaint_points = complaint_points.to_crs(nta.crs)

In [ ]:
complaint_nta = gpd.sjoin(
    complaint_points,
    nta[["NTA2020", "geometry"]],
    how="inner",
    predicate="within"
)

In [ ]:
complaint_counts = complaint_nta.groupby("NTA2020")["cmplnt_num"].nunique()
complaint_counts = complaint_counts.reset_index(name="complaint_count")

In [ ]:
comparison = complaint_counts.merge(arrest_counts, on="NTA2020")
comparison["Complaint Share"] = comparison["complaint_count"] / comparison["complaint_count"].sum() * 100
comparison["Arrest Share"] = comparison["arrest_count"] / comparison["arrest_count"].sum() * 100
comparison["Difference"] = comparison["Arrest Share"] - comparison["Complaint Share"]

In [ ]:
comparison_map = nta.merge(comparison, on="NTA2020")

In [ ]:
limit = comparison["Difference"].abs().max()

ax = comparison_map.plot(
    column="Difference",
    figsize=(15, 15),
    cmap="RdBu_r",
    vmin=-limit,
    vmax=limit,
    legend=True,
    edgecolor="black",
    linewidth=0.2,
    legend_kwds={"label": "Arrest Share Minus Complaint Share (%)"}
)

ax.set_title("Arrest Share Compared with Complaint Share", fontsize=18)
ax.axis("off")
plt.show()